# EXAMEN PRÁCTICO: EXPLORACIÓN DEL CONJUNTO DE DATOS (EDA)

## PREVIO: LIBRERÍAS Y FUNCIONES NECESARIAS

In [3]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from matplotlib import style

import seaborn as sns

import missingno as msno

%matplotlib inline

In [ ]:
import warnings
warnings.filterwarnings("ignore")

style.use('ggplot') or plt.style.use('ggplot')

### FUNCIONES

In [ ]:
def calculos_boxplot(columna, coef=1.5):
    """
    Calcula el Rango Intercuartílico (IQR) y los límites para valores considerados
    válidos (no extremos).
    -------------------------------
    Parametros de entrada:
    - columna: columna de un dataframe o Serie de Pandas
    - coef: valor del coeficiente para el cálculo de los límites válidos, por defecto 1.5
        Se puede ampliar si se quiere "relajar" la exigencia para dichos valores

    -------------------------------
    SALIDA:
    Devuelve una tupla (limite_inferior, limite_superior)
    y muestra por pantalla información

    """

    Q1, Q3 = columna.quantile([0.25,0.75])

    IQR = Q3 - Q1

    lim_inferior = Q1 - coef*IQR
    lim_superior = Q3 + coef*IQR

    print('Cuartiles: ', Q1, '\t', Q3)
    print('IQR: ',IQR)
    print('Limite inferior: ', lim_inferior)
    print('Limite superior: ', lim_superior)

    return lim_inferior, lim_superior


In [ ]:
def categoricas_unicos(dataframe, var_cat, vc_out=True):
    """
    Muestra valores unicos y conteo de las variables categóricas.
    -------------------------------
    Parametros de entrada:
    - dataframe: DF de pandas
    - var_cat: lista de variables categóricas (cadena)
    - vc_out: booleano para mostrar o no (True/False) el resultado de "value_counts()"
        Por defecto lo muestra

    -------------------------------
    SALIDA:
    No devuelve valor. Saca por pantalla la información

    """

    for discreta in var_cat:

        print(f'Variable {discreta.upper()}:')
        print('Valores unicos: ')
        print(dataframe[discreta].unique(), end='\n'*2)

        if vc_out:
            print(dataframe[discreta].value_counts(), end='\n'*2)





In [ ]:
def graf_histo_box_numericas(dataframe, var_num):
    """
    Realiza graficas histograma y boxplot de variables numéricas de un dataframe
    -------------------------------
    Parametros de entrada:
    - dataframe: DF de pandas
    - var_num: lista de variables numéricas (cadena)

    -------------------------------
    SALIDA:
    No devuelve valor. Dibuja las gráficas por pantalla

    """

    TABLEAU_CMP = ('tab:blue', 'tab:orange', 'tab:green', 'tab:red', 'tab:purple', 'tab:brown', 'tab:pink', \
                   'tab:gray','tab:olive', 'tab:cyan')

    fig, axes = plt.subplots(len(var_num), 2, \
                             figsize=(20, 5 * len(var_num)), \
                             gridspec_kw={'hspace': 0.4, 'wspace': 0.1})
    ax = axes.ravel()

    # graficas distribucion y boxplot de cada atributo
    for idx, atributo in enumerate(var_num):

        # distribucion (histograma)
        sns.histplot(dataframe[atributo], bins=30, ax=ax[2 * idx], \
                     color=TABLEAU_CMP[idx % len(TABLEAU_CMP)], \
                     alpha=0.15)

        # titulo, etiquetas histograma
        ax[2 * idx].set_title(f'HISTOGRAMA {atributo}')
        ax[2 * idx].set_xlabel(f'Valores {atributo}')
        ax[2 * idx].set_ylabel("Frequencia")

        # boxplot
        sns.boxplot(x=atributo, data=dataframe, ax=ax[2 * idx + 1], color=TABLEAU_CMP[idx % len(TABLEAU_CMP)])

        # titulo, etiquetas boxplot
        ax[2 * idx + 1].set_title(f'BOXPLOT {atributo}')
        ax[2 * idx + 1].set_xlabel(f'Valores {atributo}')



## Carga y comprobación.

Para el examen usaremos el archivo **"adult_examen.csv"** (versión reducida de un dataset conocido), que se descargará de la plataforma y se subirá donde está el notebook del examen antes de que empiece el propio examen.

En este dataset se presenta información sobre las características socioeconómicas de individuos y su ingreso anual. El significado de cada uno de los campos es el siguiente:

- **ID**: campo identificativo sin otro significado
- **age**: edad
- **workclass**: tipo de trabajo
- **marital_status**: estado civil
- **sex**: género (Male/Female)
- **hours_per_week**: horas de trabajo por semana
- **capital**: ahorros
- **income**: ingreso anual (variable objetivo categórica)



\

**Como preparación a este notebook**, inspeccionen el archivo como texto plano.


Tiene cabecera, está al inicio. Le siguen los datos. El separador es la coma.

**Cargamos los datos del archivo en un Dataframe. Comprueba la carga mostrando sólo algunas filas del DataFrame (del principio, del final o aleatorio (muestra), todas son válidas, usad sólo una)**


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
ruta = '/content/drive/MyDrive/ESPECIALISTA IA/adult_examen.csv'

df_adult = pd.read_csv(ruta)

df_adult.sample(n=6)



,ID,age,workclass,marital_status,sex,hours_per_week,capital,income
31460,31460,18,Private,Never-married,Female,25,NaN,<=50K
31208,31208,31,Private,Married-civ-spouse,Male,45,46290.084310,<=50K
17036,17036,30,Private,Married-civ-spouse,Male,40,26707.914150,<=50K
8554,8554,43,Federal-gov,Married-civ-spouse,Male,40,32338.856921,>50K
4413,4413,20,NaN,Never-married,Female,40,NaN,<=50K
5008,5008,42,Private,Married-civ-spouse,Male,40,32733.842871,<=50K


Obtengan **información general del DataFrame** mediante el **uso de "info" y "describe"


In [ ]:
df_adult.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   ID              32561 non-null  int64  
 1   age             32561 non-null  int64  
 2   workclass       30725 non-null  object 
 3   marital_status  32561 non-null  object 
 4   sex             32561 non-null  object 
 5   hours_per_week  32561 non-null  int64  
 6   capital         26150 non-null  float64
 7   income          32561 non-null  object 
dtypes: float64(1), int64(3), object(4)
memory usage: 2.0+ MB


In [ ]:

df_adult.describe()


,ID,age,hours_per_week,capital
count,32561.000000,32561.000000,32561.000000,26150.000000
mean,16280.000000,38.581647,39.435367,31159.934914
std,9399.695394,13.640433,8.482675,8087.146418
min,0.000000,17.000000,20.000000,3301.286723
25%,8140.000000,28.000000,40.000000,25919.374366
50%,16280.000000,37.000000,40.000000,29918.441921
75%,24420.000000,48.000000,45.000000,34113.812937
max,32560.000000,90.000000,50.000000,63280.743247


In [ ]:
df_adult.describe(include='object')



,workclass,marital_status,sex,income
count,30725,32561,32561,32561
unique,8,7,2,2
top,Private,Married-civ-spouse,Male,<=50K
freq,22696,14976,21790,24720


## Pregunta 1: Primeras correcciones.

Muestra las columnas del dataframe.

a) Sin entrar en gráficas ni cálculos (eso vendrá más tarde), en tu opinión, ¿todas las columnas nos pueden ser útiles para un proyecto de datos? ¿Cual eliminarías y por qué?

*(Puntuación: 0.15 ptos)*

b) Elimina duplicados y separa las columnas en variables numéricas y categóricas para trabajar más fácil con ellas.

*(Puntuación: 0.15 ptos)*

c) Siempre es bueno observar los valores de los campos de texto (variables categóricas), ver cuales son sus valores UNICOS. Utiliza una de las funciones proporcionadas arriba para conseguir esta información. ¿Ves algún error de escritura (tipogáfico)? SOLO SEÑALARLOS SI LOS HUBIERA

*(Puntuación: 0.2 ptos)*

## Pregunta 2: Análisis Unidimensional Numérico.

Dibuja las gráficas necesarias (te puedes ayudar de las funciones facilitadas al comienzo del notebook)

Interpreta los resultados (con tus palabras):

a) ¿Cómo es la distribución de cada variable numérica? Tiene "cola" a la izquierda o derecha, o por el contrario es bastante simétrica?

*(Puntuación: 0.5 ptos)*

b) ¿Presenta outliers? ¿Cómo y donde se distribuyen? (respecto de la caja)

*(Puntuación: 0.5 ptos)*

## Pregunta 3: Análisis Unidimensional Categórico.

Dibuja las gráficas necesarias (utiliza "boxplot" de seaborn)

Interpreta los resultados (con tus palabras):

a) ¿Cómo es la distribución de cada variable categórica (excepto el target)? ¿Cuántos valores únicos tiene? ¿Como es el reparto de individuos (filas) entre ellos? Si ves posibles casos donde se pueda aplicar agrupamiento, señalalo y justificalo. Escribe el código para UN SOLO caso

*(Puntuación: 1.5 ptos)*

b) ¿Cómo es la distribución del target "income"? ¿Cuántos valores únicos tiene? ¿Ves muy preocupante la diferencia entre ambas?

*(Puntuación: 0.5 ptos)*

## Pregunta 4: Análisis Bidimensional / Multidimensional.

Dibuja las gráficas necesarias. Interpreta los resultados (con tus palabras):

a) NUMÉRICO-NUMÉRICO. Utilizando las gráficas correspondientes, ¿qué puedes decir de los valores arrojados? ¿Hay variables que estén correladas entre sí? ¿Si es así, que harías?

*(Puntuación: 1.5 ptos)*

b) NUMÉRICO-CATEGÓRICO. Estudia la relación entre la variable "edad" y el target "income"
¿Sacas alguna conclusión? Razonalo

*(Puntuación: 1 ptos)*

## Pregunta 5: Estudio nulos con missingno.

Dibuja las gráficas necesarias utilizando la libreria missingno. Interpreta los resultados (con tus palabras):

a) ¿Con cuál de los métodos de dibujado de "missingno" podrías distinguir las FILAS con nulos en nuestros datos? ¿Que significa en el lado derecho de la gráfica los números que están junto al trazado?

*(Puntuación: 1 ptos)*

b) ¿Qué metodo de dibujado de "missingno" habría que utilizar para ver el PORCENTAJE de NULOS en cada COLUMNA? Utilizala en nuestra gráfica e indica la columna con mayor y la columna con menor porcentaje de nulos

*(Puntuación: 1 ptos)*

## Pregunta 6: Outliers.

Fijandonos SOLO en los outliers de "edad" y "horas de trabajo semanales":

a) Calcula el nº de outliers que hay en cada. Puedes ayudarte con las funciones que están al comienzo del notebook.

*(Puntuación: 0.75 ptos)*

b) ¿Que harías con cada uno, teniendo en cuenta también su número, cuál eliminarías y cual recortarías? Justificalo.

*(Puntuación: 0.50 ptos)*

c) Teniendo en cuenta lo que contestaste en el apartado anterior, corrige SOLO UNO DE ELLOS, EL QUE PREFIERAS.

*(Puntuación: 0.75 ptos)*